In [ ]:
# data process

In [ ]:
# model process
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

class VASONet(nn.Module):
    @staticmethod
    def init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def __init__(self):
        super().__init__()
        self.geo_mlp = nn.Sequential(
            nn.Linear(18, 320),nn.ReLU(),nn.Dropout(0.0064)
        )

        self.topo_mlp = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(352, 512), nn.ReLU(), nn.Dropout(0.0330),
            nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.0330),
            nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.0330),
            nn.Linear(512, 2)
        )
        self.apply(self.init_weights)

    def forward(self, coordinates, valence):
        x = coordinates.reshape(-1, 18).float()
        n = valence.reshape(-1, 1).int()

        hg = self.geo_mlp(x)
        ht = self.topo_mlp(n)

        h = torch.cat([hg, ht], dim=1)
        return self.decoder(h)

Valence_MLP = VASONet().to(device)
# setting
loss = nn.MSELoss()
learning_rates = 6.5e-4
epochs = 200
batch_size = 64
optimizer = torch.optim.Adam( Valence_MLP.parameters(), lr=learning_rates, betas=(0.9, 0.999) )
lr_rates = torch.optim.lr_scheduler.ExponentialLR( optimizer, gamma=0.916 )


print(Valence_MLP)
Valence_MLP.apply(VASONet.init_weights)


Using cuda device
VASONet(
  (geo_mlp): Sequential(
    (0): Linear(in_features=18, out_features=320, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.0064, inplace=False)
  )
  (topo_mlp): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=32, bias=True)
    (7): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=352, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.033, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.033, inplace=False)
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.033, inplace=False)
    (9): Linear(in_features=512, out_features=2, bias=True)
  )
)


VASONet(
  (geo_mlp): Sequential(
    (0): Linear(in_features=18, out_features=320, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.0064, inplace=False)
  )
  (topo_mlp): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=32, bias=True)
    (7): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=352, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.033, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.033, inplace=False)
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.033, inplace=False)
    (9): Linear(in_features=512, out_features=2, bias=True)
  )
)

In [ ]:
# train process 

In [ ]:
# test case result visulization